# Calcium Imaging — Analysis
Loads pipeline output: co-activity, proximity analysis, and an annotated video.

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = r"Z:\ephacoffice\DColameo\Ca_Anand_AllData\07_11_25_Min6WT_Binning2_2_250ms_Exp3\pipeline_output_single"

# Co-activity thresholds
CORR_THRESHOLD   = 0.4    # minimum Pearson r to consider two cells co-active
DIST_THRESHOLD   = 60     # maximum centroid distance (pixels) to consider cells "nearby"

# Video
VIDEO_FPS        = 5      # playback speed (frames/s); original data is TARGET_FPS
DFF_VMAX         = 1.0    # dF/F colormap saturation (adjust if traces are very large/small)
MAX_VIDEO_FRAMES = None   # int to cap video length (None = all frames)
SAVE_VIDEO_MP4   = False  # True = also write .mp4 (requires ffmpeg on PATH)
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
from pathlib import Path
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.animation as animation
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
import matplotlib.cm as cm
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from skimage.segmentation import find_boundaries
from skimage import measure
from IPython.display import HTML

plt.rcParams['figure.dpi'] = 110
plt.rcParams['animation.embed_limit'] = 512   # MB — raise if video is large

out_dir = Path(OUTPUT_DIR)

## 1 — Load pipeline outputs

In [ ]:
roi_mask  = np.load(out_dir / 'roi_mask.npy')
centroids = np.load(out_dir / 'centroids.npy')   # (n_cells, 2): [y, x]
F_raw     = np.load(out_dir / 'F_raw.npy')        # (n_cells, T)
dff       = np.load(out_dir / 'dff.npy')          # (n_cells, T)
time_axis = np.load(out_dir / 'time_axis.npy')    # (T,) seconds
mean_img  = tifffile.imread(str(out_dir / 'mean_image.tif'))
mov       = tifffile.imread(str(out_dir / 'mov_downsampled.tif')).astype(np.float32)

n_cells, T = dff.shape
H, W       = roi_mask.shape
rprops     = measure.regionprops(roi_mask)

print(f"Cells     : {n_cells}")
print(f"Frames    : {T}  ({time_axis[-1]:.0f} s)")
print(f"Frame size: {H} × {W} px")
print(f"Movie     : {mov.shape}  ({mov.nbytes/1e6:.0f} MB)")

## 2 — Overview

In [ ]:
from skimage.color import label2rgb
mean_norm = (mean_img - mean_img.min()) / (mean_img.max() - mean_img.min() + 1e-9)
overlay   = np.clip(label2rgb(roi_mask, image=mean_norm, bg_label=0, alpha=0.4), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
p1, p99 = np.percentile(mean_img, [1, 99])
axes[0].imshow(mean_img, cmap='gray', vmin=p1, vmax=p99)
axes[0].set_title('Mean image'); axes[0].axis('off')
axes[1].imshow(overlay)
axes[1].set_title(f'ROIs (n={n_cells})')
for rp in rprops:
    y, x = rp.centroid
    axes[1].text(x, y, str(rp.label), color='white', fontsize=6, ha='center', va='center')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 3 — Co-activity: pairwise correlation

In [ ]:
# Pearson correlation matrix
corr = np.corrcoef(dff)   # (n_cells, n_cells)

# Hierarchical clustering to reorder rows/cols by similarity
dist_corr = 1 - corr
np.fill_diagonal(dist_corr, 0)
dist_corr = np.clip(dist_corr, 0, None)
dist_corr = (dist_corr + dist_corr.T) / 2   # enforce exact symmetry
Z = linkage(squareform(dist_corr), method='ward')
order = dendrogram(Z, no_plot=True)['leaves']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Raw order
im = axes[0].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[0].set_title('Pairwise correlation (original order)')
axes[0].set_xlabel('Cell #'); axes[0].set_ylabel('Cell #')
plt.colorbar(im, ax=axes[0], label='Pearson r')

# Clustered order
corr_ord = corr[np.ix_(order, order)]
im2 = axes[1].imshow(corr_ord, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1].set_title('Pairwise correlation (clustered)')
axes[1].set_xlabel('Cell # (reordered)'); axes[1].set_ylabel('Cell # (reordered)')
plt.colorbar(im2, ax=axes[1], label='Pearson r')

plt.tight_layout()
plt.show()

n_coactive = int(((corr > CORR_THRESHOLD).sum() - n_cells) / 2)
print(f"Pairs with r > {CORR_THRESHOLD}: {n_coactive}")

## 4 — Proximity vs co-activity

In [ ]:
# Pairwise Euclidean distance between centroids
dist_mat = squareform(pdist(centroids))   # (n_cells, n_cells) pixels

# Upper triangle only (no diagonal)
triu_idx = np.triu_indices(n_cells, k=1)
pairwise_dist = dist_mat[triu_idx]
pairwise_corr = corr[triu_idx]

# Classify pairs
nearby    = pairwise_dist < DIST_THRESHOLD
coactive  = pairwise_corr > CORR_THRESHOLD
both      = nearby & coactive

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scatter: distance vs correlation
colors = np.where(both, 'crimson', np.where(nearby, 'orange', np.where(coactive, 'steelblue', 'lightgray')))
axes[0].scatter(pairwise_dist, pairwise_corr, c=colors, s=12, alpha=0.6, linewidths=0)
axes[0].axhline(CORR_THRESHOLD, color='steelblue', lw=1, ls='--', label=f'r = {CORR_THRESHOLD}')
axes[0].axvline(DIST_THRESHOLD, color='orange',    lw=1, ls='--', label=f'd = {DIST_THRESHOLD} px')
axes[0].set_xlabel('Centroid distance (px)')
axes[0].set_ylabel('Pearson r')
axes[0].set_title('Proximity vs co-activity (all pairs)')
legend_els = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='crimson',    ms=7, label='nearby + co-active'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='orange',     ms=7, label='nearby only'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='steelblue',  ms=7, label='co-active only'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='lightgray',  ms=7, label='neither'),
]
axes[0].legend(handles=legend_els, fontsize=8)

# Distance matrix heatmap
im = axes[1].imshow(dist_mat, cmap='viridis_r', aspect='auto')
axes[1].set_title('Centroid distance matrix (px)')
axes[1].set_xlabel('Cell #'); axes[1].set_ylabel('Cell #')
plt.colorbar(im, ax=axes[1], label='Distance (px)')

plt.tight_layout()
plt.show()

print(f"Nearby pairs (d < {DIST_THRESHOLD} px)               : {nearby.sum()}")
print(f"Co-active pairs (r > {CORR_THRESHOLD})               : {coactive.sum()}")
print(f"Nearby AND co-active                              : {both.sum()}")

## 5 — Co-active neighbour map
Lines connect pairs that are both nearby and co-active. Line colour = correlation, width ∝ correlation.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10 * H / W))
p1, p99 = np.percentile(mean_img, [1, 99])
ax.imshow(mean_img, cmap='gray', vmin=p1, vmax=p99)

edge_cmap = cm.get_cmap('YlOrRd')
edge_norm = Normalize(vmin=CORR_THRESHOLD, vmax=1.0)

# Draw edges for nearby+co-active pairs
i_idx, j_idx = triu_idx
for i, j, d, r in zip(i_idx, j_idx, pairwise_dist, pairwise_corr):
    if d < DIST_THRESHOLD and r > CORR_THRESHOLD:
        y0, x0 = centroids[i]
        y1, x1 = centroids[j]
        ax.plot([x0, x1], [y0, y1],
                color=edge_cmap(edge_norm(r)),
                lw=1 + 3 * edge_norm(r),
                alpha=0.8, solid_capstyle='round')

# Draw cell dots
for i, rp in enumerate(rprops):
    y, x = rp.centroid
    ax.plot(x, y, 'o', color='white', ms=4, mew=0)
    ax.text(x + 3, y, str(rp.label), color='white', fontsize=6, va='center')

sm = cm.ScalarMappable(cmap=edge_cmap, norm=edge_norm)
plt.colorbar(sm, ax=ax, label='Pearson r', shrink=0.5)
ax.set_title(f'Co-active neighbours  (r > {CORR_THRESHOLD}, d < {DIST_THRESHOLD} px)')
ax.axis('off')
plt.tight_layout()
plt.show()

## 6 — Annotated video
Play the MP4 generated by `ca_pipeline.ipynb` (Step 11). No raw data reload needed.

In [ ]:
from IPython.display import Video

mp4_path = out_dir / 'ca_video.mp4'
assert mp4_path.exists(), f"Video not found: {mp4_path}\nRun Step 11 in ca_pipeline.ipynb first."
print(f"Playing: {mp4_path}  ({mp4_path.stat().st_size/1e6:.1f} MB)")
Video(str(mp4_path), embed=True, width=900)